In [ ]:
# !pip install torch==2.5.0
# !pip install numpy==1.26.4
# !pip install openai==1.79.0
# !pip install tenacity==9.1.2
# !pip install tiktoken==0.9.0
# !pip install transformers==4.51.3
# !pip install pandas==2.2.3
# !pip install scikit-learn==1.6.1
# !pip install bitsandbytes==0.45.5
# !pip install datasets==3.6.0
# !pip install sentencepiece==0.2.0
# !pip install peft==0.15.2
# !pip install evaluate==0.4.3
# !pip install trl==0.11.4
# !pip install protobuf==6.31.0
# !pip install python-dotenv==1.1.0
# !pip install pandas_ta
# !pip install ollama==0.4.8
# !pip install accelerate==1.7.0
# !pip install ipywidgets
# !pip install pynvml==8.1.7
# !pip uninstall torch torchvision torchaudio -y
# !pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124
# !pip install --upgrade setuptools
# !pip install torchdiffeq


In [4]:
import os
import pandas as pd
import pandas_ta as ta  # Sử dụng pandas_ta vì dễ cài đặt và tích hợp với pandas
import numpy as np
import argparse

args = argparse.Namespace(
    price_dir="Data/sample/top_1_stock/",  # Thư mục dữ liệu giá
    price_raw_dir="Data/price/raw/",  # Thư mục dữ liệu giá Data\price\raw
    tweet_dir="Data/tweet/raw/",  # Thư mục dữ liệu tweet
    seq_len=5,  # Độ dài chuỗi đầu vào
    technical_indicator_dir="Data/price/technical_indicator/",
    llm_summarize="OpenAILLM", # OpenAILLM // DeepSeekLLM
    summarized = 'Data/summarized',
)


### Tính chỉ báo kỹ thuật và Lưu vào các file csv

In [ ]:
## Tính Chỉ Báo Kỹ Thuật

def calculate_technical_indicators(data):
    """
    Tính toán các chỉ số kỹ thuật phổ biến cho dữ liệu chứng khoán.

    Args:
        data (pandas.DataFrame): DataFrame chứa dữ liệu chứng khoán (Open, High, Low, Close, Volume).
                                 Yêu cầu các cột 'Open', 'High', 'Low', 'Close', 'Volume'.

    Returns:
        pandas.DataFrame: DataFrame chứa dữ liệu gốc và các chỉ số kỹ thuật.
    """

    # 1. Moving Averages
    data['SMA_5'] = ta.sma(data['Close'], length=5)
    data['EMA_5'] = ta.ema(data['Close'], length=5)

    # 2. MACD
    macd = ta.macd(data['Close'], fast=12, slow=26, signal=9)
    data['MACD'] = macd['MACD_12_26_9']  # Lấy đường MACD
    data['MACD_SIGNAL'] = macd['MACDs_12_26_9']  # Lấy đường tín hiệu
    data['MACD_HIST'] = macd['MACDh_12_26_9']  # Lấy histogram

    # 3. RSI
    data['RSI'] = ta.rsi(data['Close'], length=5)

    # 4. Bollinger Bands
    bbands = ta.bbands(data['Close'], length=20, std=2)
    data['BB_UPPER'] = bbands['BBU_20_2.0']
    data['BB_LOWER'] = bbands['BBL_20_2.0']
    data['BB_MIDDLE'] = bbands['BBM_20_2.0']

    # 5. Volume Indicators (OBV)
    data['OBV'] = ta.obv(data['Close'], data['Volume'])

    # 6. ADX
    adx = ta.adx(data['High'], data['Low'], data['Close'], length=14)
    data['ADX'] = adx['ADX_14']
    data['DMP'] = adx['DMP_14']  # Positive Directional Movement
    data['DMN'] = adx['DMN_14']  # Negative Directional Movement

    data = data.drop(columns = ["Open","High","Low","Close", "Adj Close","Volume"]) 
    return data.round(2)


folder_path = args.price_raw_dir
technical_indicator_folder = args.technical_indicator_dir  # Thư mục lưu file sau khi xử lý Data\price\technical_indicator
# Tạo thư mục technical_indicator_folder nếu chưa có
os.makedirs(technical_indicator_folder, exist_ok=True)


# Kiểm tra thư mục có tồn tại không
if os.path.exists(folder_path):
    # Duyệt qua từng file trong thư mục
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        technical_indicator_path = os.path.join(technical_indicator_folder, file)  # Đường dẫn file mới
        # Kiểm tra nếu file là CSV
        if file.endswith('.csv'):
            print(f"Đang mở: {file}")

            # Đọc file CSV vào DataFrame
            df = pd.read_csv(file_path)
            # print(df.head())  # In 5 dòng đầu của file

            df = calculate_technical_indicators(df)

            # Ghi file vào thư mục processed
            df.to_csv(technical_indicator_path, index=False)
            print(f"Đã lưu: {technical_indicator_path}")
        # break
else:
    print("Thư mục không tồn tại.")

### Chọn dữ liệu để train và test
- Không dự đoán các ngày liên tục, do seq_len=5, nên để tiếp kiệm thời gian chạy. Chỉ dự đoán 5 ngày 1 lần 
- Các ngày được dự đoán sẽ từ '2020-02-19' trở đi vì chúng đẩy đủ các chỉ báo báo kĩ thuật (technical indicator)

In [11]:
import os
import pandas as pd

def process_price_files(company_names, folder_path, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for name in company_names:
        file_path = os.path.join(folder_path, f"{name}.txt")
        if os.path.exists(file_path):
            # Đọc file .txt, cách nhau bằng tab, không có header
            df = pd.read_csv(file_path, sep='\t', header=None)

            # Đặt tên cột
            df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

            # Lọc từ ngày 2020-02-19 trở đi
            df['Date'] = pd.to_datetime(df['Date'])
            df = df[df['Date'] >= '2020-02-19']

            # Cứ 5 dòng lấy 1 dòng
            df = df.iloc[::5].reset_index(drop=True)

            # Ghi lại file .txt
            output_path = os.path.join(output_folder, f"{name}.txt")
            with open(output_path, 'w') as f:
                for _, row in df.iterrows():
                    line = f"{row['Date'].strftime('%Y-%m-%d')}\t"
                    line += "\t".join(f"{row[col]:.6f}" for col in ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'])
                    f.write(line + "\n")

            print(f"Đã xử lý xong và ghi vào: {output_path}")
        else:
            print(f"Không tìm thấy file: {file_path}")


In [ ]:
company_names_top1_stock = ['AAPL','AMZN', 'AMT', 'BHP','BRK-A','WMT', 'NEE', 'XOM', 'UNH', 'GOOG', 'UPS']

folder_path = os.path.join('Data', 'price', 'preprocessed')
output_folder = os.path.join('Data', 'sample', 'top_1_stock')

process_price_files(company_names_top1_stock, folder_path, output_folder)


#### Phần này dành cho remain stock 

In [12]:
# import os

# def get_remaining_companies(price_folder, company_names, extension='.txt'):
#     # Lấy danh sách tất cả các file trong thư mục
#     all_files = os.listdir(price_folder)

#     # Lọc lấy tên công ty từ file có đúng phần mở rộng
#     all_company_names = [
#         os.path.splitext(filename)[0]
#         for filename in all_files
#         if filename.endswith(extension)
#     ]

#     # Loại bỏ các tên đã có sẵn trong company_names
#     remaining = [name for name in all_company_names if name not in company_names]
#     return remaining

# company_names_top1_stock = ['AAPL','AMZN', 'AMT', 'BHP','BRK-A','WMT', 'NEE', 'XOM', 'UNH', 'GOOG', 'UPS']

# price_folder = os.path.join('Data', 'price', 'preprocessed')  # hoặc 'Data/price/raw', tuỳ bạn

# company_names_remaining = get_remaining_companies(price_folder, company_names_top1_stock, extension='.txt')
# print("Các công ty còn lại:", company_names_remaining)

# output_folder_remaining = os.path.join('Data', 'sample', 'remain_stock')

# process_price_files(company_names_remaining, folder_path, output_folder_remaining)

### TÓM TẮT DỮ LIỆU (SUMMARIZE)

In [ ]:
from data_load.dataloader import DataLoader

print('Args in experiment:')
print(args)

#===========================================================================================================================================
# Collect demonstration data
print("Loading Train Agents...")
dataloader = DataLoader(args)
data = dataloader.load(flag="train")
# print(data)

summarized_path = args.summarized
os.makedirs(summarized_path, exist_ok=True)

# Data\summarized
path = os.path.join(summarized_path, args.llm_summarize + "_top1_stock_data_train_sample.csv")
path

# Lưu DataFrame vào tệp CSV
data.to_csv(path, index=False)  # index=False để không lưu chỉ số dòng
print(f"DataFrame đã được lưu vào '{path}'")


In [ ]:
from data_load.dataloader import DataLoader

print('Args in experiment:')
print(args)

#===========================================================================================================================================
# Collect demonstration data
print("Loading Train Agents...")
dataloader = DataLoader(args)
data_test = dataloader.load(flag="test")
# print(data)

summarized_path = args.summarized
os.makedirs(summarized_path, exist_ok=True)

# Data\summarized
path = os.path.join(summarized_path, args.llm_summarize + "_top1_stock_data_test.csv")
path

# Lưu DataFrame vào tệp CSV
data_test.to_csv(path, index=False)  # index=False để không lưu chỉ số dòng
print(f"DataFrame đã được lưu vào '{path}'")


### Gắn Dữ liệu chỉ báo kỹ thuật vào dữ liệu tin tức

In [6]:

def format_technical_indicators(row):
    """
    Chuyển đổi một dòng của DataFrame thành văn bản, bỏ qua giá trị NaN.

    Args:
        row (pd.Series): Một dòng của DataFrame chứa các chỉ số kỹ thuật.

    Returns:
        str: Văn bản chứa chỉ số kỹ thuật, mỗi dòng cách nhau bởi '\n'.
    """
    lines = []
    
    for col, value in row.items():
        if pd.notna(value):  # Bỏ qua nếu giá trị là NaN
            lines.append(f"{col}: {value:,}")  # Định dạng số với dấu phẩy
    
    return "\n".join(lines)  # Kết hợp các dòng thành văn bản


def add_technical_indicator(data, technical_indicator_dir):

    col_to_drop = "technical_indicator"

    if col_to_drop in data.columns:
        data = data.drop(columns=[col_to_drop])

    # Lấy danh sách các giá trị duy nhất của cột "ticker"
    unique_tickers = data["ticker"].unique()
    unique_tickers

    # DataFrame để lưu kết quả cuối cùng
    final_df = pd.DataFrame()


    for file_name in unique_tickers:

        technical_indicator_path = os.path.join(technical_indicator_dir, file_name + ".csv")

        df_technical_indicator = pd.read_csv(technical_indicator_path)

        # Loại bỏ cột 'Date' để chỉ giữ các chỉ báo kỹ thuật
        df_technical_indicator["technical_indicator"] = df_technical_indicator.drop(columns=["Date"]).apply(format_technical_indicators, axis=1)

        # Thêm cột "ticker" với giá trị file_name
        df_technical_indicator["ticker"] = file_name

        # Hiển thị kết quả  
        df_technical_indicator = df_technical_indicator[["ticker", "Date", "technical_indicator"]].rename(columns={"Date": "date"})
        # Merge theo 2 cột "ticker" và "Date"
        merged_df = pd.merge(data, df_technical_indicator, on=["ticker", "date"], how="inner")

        # Cộng dồn kết quả
        final_df = pd.concat([final_df, merged_df], ignore_index=True)

    return final_df

    
# Đọc file CSV vào DataFrame
summarized_path = args.summarized

#========================================= TRAIN =========================================================

# Data\summarized
path_train = os.path.join(summarized_path, args.llm_summarize + "_top1_stock_data_train_sample.csv")
data_train = pd.read_csv(path_train)


technical_indicator_dir = args.technical_indicator_dir
data_train = add_technical_indicator(data_train, technical_indicator_dir)
data_train

data_train.to_csv(path_train, index=False)


#========================================= TEST =========================================================
# Data\summarized
path_test = os.path.join(summarized_path, args.llm_summarize + "_top1_stock_data_test.csv")
data_test = pd.read_csv(path_test)


technical_indicator_dir = args.technical_indicator_dir
data_test = add_technical_indicator(data_test, technical_indicator_dir)
data_test

data_test.to_csv(path_test, index=False)

